# S6 — Tiles and web map

Stage 6 of the Manhattan Sidewalk Shade Index pipeline.

Reprojects the final shade-index layer to EPSG:4326, adds a `tree_count_nearby` attribute (trees within 15 m of each unit — the click-info "trees contributing" count CLAUDE.md §5 asks for), and exports GeoJSON for tippecanoe. **tippecanoe itself runs from WSL/Ubuntu via pipx, not conda** (per `modern-gis/SKILL.md` §7/§10) — invoked separately after this notebook, not from within it.

**Accept when:** the map loads from a clean clone following README steps only, the slider visibly rotates the shade pattern east→west across the day, and tile size is reasonable.

In [1]:
from pathlib import Path
from datetime import datetime

import yaml
import duckdb
import geopandas as gpd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
OUTPUT_CRS = config["output_crs"]
print("s6: Build tiles and web map (data prep)")
print(f"Timestamp: {datetime.now().isoformat()}\n")

s6: Build tiles and web map (data prep)
Timestamp: 2026-08-28T14:17:47.228451



## Add `tree_count_nearby`, reproject, export GeoJSON

In [2]:
final = gpd.read_parquet(PROJECT_ROOT / config["output"]["shade_index_units"])
trees = gpd.read_parquet(PROJECT_ROOT / config["output"]["ingested_trees"])
print(f"units: {len(final):,}  trees: {len(trees):,}")

units_scratch = PROJECT_ROOT / config["interim_dir"] / "_scratch_units_s6.parquet"
trees_scratch = PROJECT_ROOT / config["interim_dir"] / "_scratch_trees_s6.parquet"
final[["unit_id", "geometry"]].to_parquet(units_scratch)
trees[["geometry"]].to_parquet(trees_scratch)

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute(f"CREATE TABLE units AS SELECT unit_id, geometry AS geom FROM read_parquet('{units_scratch.as_posix()}')")
con.execute(f"CREATE TABLE trees AS SELECT geometry AS geom FROM read_parquet('{trees_scratch.as_posix()}')")
tree_counts = con.execute("""
    SELECT u.unit_id, COUNT(t.geom) AS tree_count_nearby
    FROM units u LEFT JOIN trees t ON ST_DWithin(u.geom, t.geom, 15.0)
    GROUP BY u.unit_id
""").fetchdf()
units_scratch.unlink()
trees_scratch.unlink()

final = final.merge(tree_counts, on="unit_id")
print(final["tree_count_nearby"].describe())

units: 34,603  trees: 62,416


count    34602.000000
mean         4.382868
std          4.564714
min          0.000000
25%          0.000000
50%          3.000000
75%          7.000000
max         33.000000
Name: tree_count_nearby, dtype: float64


In [3]:
final_4326 = final.to_crs(OUTPUT_CRS)

geojson_path = PROJECT_ROOT / config["output"]["shade_index_geojson"]
geojson_path.parent.mkdir(parents=True, exist_ok=True)
final_4326.to_file(geojson_path, driver="GeoJSON")
print(f"Wrote {geojson_path} ({geojson_path.stat().st_size:,} bytes, {len(final_4326):,} features)")

assert final_4326.crs.to_epsg() == 4326
assert geojson_path.stat().st_size > 0
print("\ns6 data prep complete. Next: run tippecanoe from WSL to build web/tiles/shade_index.pmtiles.")

C:\Users\juanz\miniconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Wrote C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\processed\shade_index_units.geojson (78,243,913 bytes, 34,602 features)

s6 data prep complete. Next: run tippecanoe from WSL to build web/tiles/shade_index.pmtiles.
